Git clone the repo and install the requirements. (ignore the pip errors about protobuf)

In [1]:
#@title Environment Setup

from pathlib import Path

OPTIONS = {}

WORKSPACE = 'ComfyUI'
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}

OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /

    from google.colab import drive
    drive.mount('/content/drive')

    WORKSPACE = "/content/drive/MyDrive/ComfyUI"

    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo "-= Initial setup ComfyUI (Original)=-" && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo "-= Updating ComfyUI =-"
  !git pull
  !rm "/content/drive/MyDrive/ComfyUI/custom_nodes/comfyui-impact-pack.py"

%cd custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack
%cd $WORKSPACE

!echo -= Install dependencies =-
!pip -q install xformers -r requirements.txt


Mounting Google Drive...
/
Mounted at /content/drive
/content/drive/MyDrive
/content/drive/MyDrive/ComfyUI
-= Updating ComfyUI =-
Already up to date.
rm: cannot remove '/content/drive/MyDrive/ComfyUI/custom_nodes/comfyui-impact-pack.py': No such file or directory
/content/drive/MyDrive/ComfyUI/custom_nodes
fatal: destination path 'ComfyUI-Impact-Pack' already exists and is not an empty directory.
/content/drive/MyDrive/ComfyUI
-= Install dependencies =-


In [2]:
from huggingface_hub import hf_hub_download, snapshot_download
import shutil, os

COMFY_DIR = "/content/drive/MyDrive/ComfyUI"
MODELS_DIR = f"{COMFY_DIR}/models"

# 1. Download Native ComfyUI MOSS-Image BF16 Stack
print("Downloading MOSS-Image SFT (bf16), Text Encoder, and VAE...")
hf_hub_download(
    repo_id="Comfy-Org/MOSS-Image",
    filename="diffusion_models/moss_image_sft_bf16.safetensors",
    local_dir=MODELS_DIR
)
hf_hub_download(
    repo_id="Comfy-Org/MOSS-Image",
    filename="text_encoders/qwen3.5_9b_bf16.safetensors", # Swap to qwen3.5_9b_int8_convrot.safetensors if low on Drive storage
    local_dir=MODELS_DIR
)
hf_hub_download(
    repo_id="Comfy-Org/MOSS-Image",
    filename="vae/flux2-vae_small.safetensors",
    local_dir=MODELS_DIR
)

# 2. Download Rapid-MOSS 4-Step Adapter & Custom Node
print("Downloading Rapid-MOSS 4-step adapter and custom node...")
rapid_tmp = "/content/rapid_moss_tmp"
snapshot_download(repo_id="darask0/MOSS-Image_4-step_adapter_lora", local_dir=rapid_tmp)

# Install the custom node into ComfyUI/custom_nodes/rapid-moss
node_dst = f"{COMFY_DIR}/custom_nodes/rapid-moss"
if os.path.exists(node_dst):
    shutil.rmtree(node_dst)
shutil.copytree(f"{rapid_tmp}/comfyui/rapid-moss", node_dst)

# Move the 3 adapter weights into ComfyUI/models/rapid_moss/rapid-moss-4step/
adapter_dst = f"{MODELS_DIR}/rapid_moss/rapid-moss-4step"
os.makedirs(adapter_dst, exist_ok=True)
for f in ["lora.safetensors", "time_embedding.safetensors", "dataspace.safetensors"]:
    shutil.copy(f"{rapid_tmp}/{f}", f"{adapter_dst}/{f}")

# 3. Download Image-to-Image / Edit LoRAs (for Img2Img workflows)
print("Downloading Moss-Image-Edit & 8-Step DMD LoRAs...")
hf_hub_download(
    repo_id="chinoll/Moss-Image-Edit",
    filename="adapter_model.safetensors",
    local_dir=f"{MODELS_DIR}/loras"
)
shutil.move(f"{MODELS_DIR}/loras/adapter_model.safetensors", f"{MODELS_DIR}/loras/moss_image_edit.safetensors")

hf_hub_download(
    repo_id="lodestones/MOSS-Image-SFT-8-steps-dmd",
    filename="lora.safetensors",
    local_dir=f"{MODELS_DIR}/loras"
)
shutil.move(f"{MODELS_DIR}/loras/lora.safetensors", f"{MODELS_DIR}/loras/moss_sft_8step_dmd.safetensors")

print("Setup complete! Ready to launch ComfyUI.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 50 files:   0%|          | 0/50 [00:00<?, ?it/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  297MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

lora.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

lora.safetensors: downloading bytes:           |  0.00B            

Setup complete! Ready to launch ComfyUI.


### Run ComfyUI with localtunnel (Recommended Way)




In [ ]:
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)")
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server --temp-directory /content/temp

⠙
changed 22 packages in 227ms
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend eager: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'convrot_w4a4_linear', 'dequantize_convrot_w4a4_weight', 'dequantize_int8_convrot_weight', 'dequantize_int8_convrot_weight_dtype', 'dequantize_int8_embedding', 'dequantize_int8_simple', 'dequantize_int8_simple_dtype', 'dequantize_mxfp8', 'dequantize

### Run ComfyUI with colab iframe (use only in case the previous way with localtunnel doesn't work)

You should see the ui appear in an iframe. If you get a 403 error, it's your firefox settings or an extension that's messing things up.

If you want to open it in another window use the link.

Note that some UI features like live image previews won't work because the colab iframe blocks websockets.

In [ ]:
import threading
import time
import socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  from google.colab import output
  output.serve_kernel_port_as_iframe(port, height=1024)
  print("to open it in a window you can open this link here:")
  output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server